# Preprocess data to prepare for learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
try:
    nltk.data.find('corpora/stopwords')
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')
import ast
from nltk.stem import PorterStemmer

Load data

In [ ]:
# Load data
df = pd.read_csv('../data/unfiltered/active_authors_time_windows/raw/active_authors_exclude_1month.csv')
print(f"Loaded {len(df)} author documents")
print(f"Columns: {df.columns.tolist()}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Average text length: {df['concatenated_text'].str.len().mean():.0f} characters")

# filename prefix 
filename = "exclude_1month"  

1. Convert to lower case

In [ ]:
# Convert to lowercase
df['concatenated_text'] = df['concatenated_text'].str.lower()
print(f"\nSample (first 200 chars):\n{df['concatenated_text'].iloc[0][:200]}...")

# Save to CSV
df.to_csv(f'../data/filtered/concatenated/preprocessed/{filename}_lower.csv', index=False)

2. Expanding contractions

In [ ]:
# Dictionary of common contractions
contractions_dict = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "how'd": "how did",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'll": "i will",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'll": "it will",
    "it's": "it is",
    "let's": "let us",
    "shouldn't": "should not",
    "that's": "that is",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'll": "we will",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what's": "what is",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have"
}

# Function to expand contractions
def expand_contractions(text):
    for contraction, expansion in contractions_dict.items():
        text = re.sub(r'\b' + contraction + r'\b', expansion, text, flags=re.IGNORECASE)
    return text

# Normalise apostrophes 
df['concatenated_text'] = df['concatenated_text'].str.replace("’", "'", regex=False)

# Apply contraction expansion
df['concatenated_text'] = df['concatenated_text'].apply(expand_contractions)
print(f"\nSample (first 200 chars):\n{df['concatenated_text'].iloc[0][:200]}...")

# Save to CSV
df.to_csv(f'../data/unfiltered/time_windows/preprocessed/{filename}_expanded.csv', index=False)

3: Tokenisation

In [ ]:
# Function to tokenize text
def tokenize(text):
    # Tokenize text into words
    tokens = word_tokenize(text)
    return tokens

# Apply tokenisation
df['tokens'] = df['concatenated_text'].apply(tokenize)
df['tokenized_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Update concatenated_text to be the tokenized version
df['concatenated_text'] = df['tokenized_text']

print(f"Sample tokens: {df['tokens'].iloc[0][:20]}")
print(f"Average tokens per document: {df['tokens'].apply(len).mean():.0f}")

# Save to CSV
df.to_csv(f'../data/unfiltered/time_windows/preprocessed/{filename}_tokenized.csv', index=False)

4. Remove punctuation

In [ ]:
import string

# Function to remove punctuation from tokens
def remove_punctuation(tokens):
    # Keep only tokens that contain at least one alphanumeric character
    tokens = [token for token in tokens if any(c.isalnum() for c in token)]
    return tokens

# Apply punctuation removal
df['tokens'] = df['tokens'].apply(remove_punctuation)
df['tokenized_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Update concatenated_text to be the punctuation-removed version
df['concatenated_text'] = df['tokenized_text']

print(f"Sample tokens: {df['tokens'].iloc[0][:20]}")
print(f"Average tokens per document: {df['tokens'].apply(len).mean():.0f}")

# Save to CSV
df.to_csv(f'../data/filtered/concatenated/preprocessed/{filename}_nopunct.csv', index=False)

5a: Lemmatisation

In [ ]:
# Initialise lemmatizer
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    """Map NLTK POS tag to WordNet POS tag."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN 

def lemmatize_tokens(tokens):
    # POS-tag first then lemmatise with the correct POS
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(token, get_wordnet_pos(tag))
            for token, tag in tagged]

# Apply lemmatization
df['tokens'] = df['tokens'].apply(lemmatize_tokens)
df['tokenized_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Update concatenated_text to be the lemmatized version
df['concatenated_text'] = df['tokenized_text']

print(f"Sample tokens: {df['tokens'].iloc[0][:20]}")
print(f"Average tokens per document: {df['tokens'].apply(len).mean():.0f}")

# Save to CSV
df.to_csv(f'../data/unfiltered/active_authors_time_windows/preprocessed/{filename}_lemmatized.csv', index=False)

5b: Stemming 

In [ ]:
# Reload data from punctuation-cleaned version 
df = pd.read_csv(f'../data/unfiltered/active_authors_time_windows/preprocessed/{filename}_nopunct.csv')

# Convert string representation back to actual lists
df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

stemmer = PorterStemmer()

# Function to stem tokens
def stem_tokens(tokens):
    # Convert each token to its stem
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

# Apply stemming
df['tokens'] = df['tokens'].apply(stem_tokens)
df['tokenized_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Update concatenated_text to be the final stemmed version
df['concatenated_text'] = df['tokenized_text']

print(f"Sample tokens: {df['tokens'].iloc[0][:20]}")
print(f"Average tokens per document: {df['tokens'].apply(len).mean():.0f}")
print(f"\nSample processed text (first 200 chars):\n{df['concatenated_text'].iloc[0][:200]}...")

# Save to CSV
df.to_csv(f'../data/unfiltered/active_authors_time_windows/preprocessed/{filename}_stemmed.csv', index=False)